# Simulations

This notebook assess errors in the GPROF database simulations.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [2]:
from gprof_nn.plotting import set_style
set_style()

## Data

We use collocation between ATMS and GPM CMB and match them with the corresponding GPROF simulator files.


In [6]:
collocations = sorted(list(Path("/edata1/simon/gprof_v8/collocations/combined/amsr2/gridded/").glob("20181005*.nc")))

In [7]:
collocations[:10]

[]

In [ ]:
import hdf5plugin
from pyresample.geometry import SwathDefinition

colloc_ind = 224
atms_observations = xr.load_dataset(collocations[colloc_ind], group="input_data")
reference_data = xr.load_dataset(collocations[colloc_ind], group="reference_data")

def get_granule(reference_data: xr.Dataset) -> int:
    """
    Extract granule from collocation attributes.

    Args:
        reference_data: An xarray.Dataset containing the collocation reference data

    Return:
        The granule number as an integer.
    """
    l1c_file = reference_data.attrs["input_files"]
    granule = int(l1c_file.split(".")[-3])
    return granule

granule = get_granule(reference_data)
lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
area = SwathDefinition(lats=lats, lons=lons)

In [ ]:
from gprof_nn.data.sim import SimFile
from pansat.utils import resample_data
from scipy.interpolate import interpn

from gprof_nn import sensors
from gprof_nn.data.sim import collocate_targets, SimulatorInput
from gprof_nn.data.sim import simulate_tbs_satformer
from gprof_nn.data.utils import decompress_scene


def find_sim_file(granule: int) -> Path:
    """
    Find the sim file for a given granule.

    Args:
        granule: The granule number as an integer

    Return:
        A path object pointing to the sim file corresopnding to the given granule.
    """
    pattern = f"**/*.{granule:06}.sim"
    sim_files = sorted(list(Path("/qdata1/pbrown/dbaseV8/simV8_amsr2/").glob(pattern)))
    return next(iter(sim_files))

def load_and_resample_sim_data(granule, amsr2_observations, area):
    """
    Load and resample simulator data for a given granule and resample to the given geometry.

    Args:
        granule: The granule number as an integer.
        amsr2_observations: The collocated AMSR2 observations to use to interpolate the simulated brightness temperatures.
        area: A pyresample area definition defining the grid to which to resample the simulator data.

    Return:
        An xarray.Dataset containing the resample simulator data.
    """
    sim_file_path = find_sim_file(granule)
    sim_file = SimFile(sim_file_path)

    sim_data = collocate_targets(sim_file_path, sensors.AMSR2, "/qdata2/archive/ERA5/")
    
    lons, lats = area.get_lonlats()
    lon_min = lons.min()
    lon_max = lons.max()
    lat_min = lats.min()
    lat_max = lats.max()
    scan_mask = (
        (lon_min <= sim_data.longitude.data) * (sim_data.longitude.data <= lon_max) *
        (lat_min <= sim_data.latitude.data) * (sim_data.latitude.data <= lat_max)
    ).any(-1)
    sim_data = sim_data[{"scans": scan_mask}]
    scan_time = sim_data.scan_time.data
    float_vars = [
        "latitude", "longitude", "brightness_temperatures", "total_column_water_vapor", "two_meter_temperature",
        "moisture_convergence", "leaf_area_index", "snow_depth", "land_fraction", "ice_fraction", "elevation",
        "earth_incidence_angle", "simulated_brightness_temperatures", "brightness_temperature_biases", "surface_type",
        "surface_precip"
    ]

    sim_data = decompress_scene( sim_data, float_vars + ["scan_time", "angles"])
    
    #simulate_tbs_satformer("../satformer.ckpt", sim_data, sensors.AMSR2)
    sim_data = sim_data.rename(simulated_brightness_temperatures="simulated_tbs")

    tbs_sim = sim_data["simulated_tbs"].data
    biases = sim_data["brightness_temperature_biases"].data
    valid = (tbs_sim >= 0.0) * (np.abs(biases) < 50)
    tbs_sim[~valid] = np.nan
    biases[~valid] = np.nan

    sim_data_r = resample_data(sim_data, area, new_dims=("latitude", "longitude"), radius_of_influence=15e3)
    angles = sim_file.header[0][3]
    sim_data_r = sim_data_r.assign_coords(angles=angles).sortby("angles")
    
    return sim_data_r

In [ ]:
sim_files = sorted(list(Path("/qdata1/pbrown/dbaseV8/simV8_amsr2/").glob("**/*.sim")))

## Case study

In [ ]:
import os
os.environ["PANSAT_PASSWORD"]="not_a_secret"

In [ ]:
from gprof_nn.sensors import AMSR2
AMSR2.kind

In [ ]:
AMSR2.frequencies

In [ ]:
data = xr.load_dataset("/edata2/simon/gprof_v8/satformer/training_data/gmi_amsr2_20200210120857_20200210122120_0000.nc")

In [ ]:
data.target_meta_data.mean(axis=(-2, -1)).data[:, 5]

In [ ]:
sim_file = SimFile("/qdata1/pbrown/dbaseV8/simV8_amsr2/1810/AMSR2.dbsatTb.20181001.026079.sim")
sim_data = sim_file.to_xarray_dataset()

In [ ]:
from gprof_nn.data.sim import BEAM_WIDTHS
BEAM_WIDTHS

In [ ]:
AMSR2.viewing_geometry.altitude

In [ ]:
data = xr.load_dataset("/gdata1/simon/gprof_v8/satformer/training_data/amsr2_gmi_20200206172708_20200206173257_0000.nc")
data.input_meta_data.mean(("pixels", "scans"))[:, 6]

In [ ]:
colloc_ind = 9
colloc = collocations[colloc_ind]

atms_observations = xr.load_dataset(colloc, group="input_data")
lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
area = SwathDefinition(lats=lats, lons=lons)

reference_data = xr.load_dataset(colloc, group="reference_data")
granule = get_granule(reference_data)
    
sim_data_r = load_and_resample_sim_data(granule, atms_observations, area)

In [ ]:
import cartopy.crs as ccrs
from matplotlib.colors import to_hex
from gprof_nn.plotting import add_ticks
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
crs = ccrs.PlateCarree()

fig = plt.figure(figsize=(12,8))
ax = fig.add_subplot(1, 1, 1, projection=crs)

lons = sim_data_r.longitude.data
lats = sim_data_r.latitude.data
ax.pcolormesh(lons, lats, sim_data_r.brightness_temperatures[..., 3], cmap="magma")
swath = to_hex("C0")
swath = swath + "88"
ax.contourf(lons, lats, np.isfinite(sim_data_r.simulated_tbs[..., 3]), colors=["#00000000", swath])

ax.contour(lons, lats, np.isfinite(atms_observations.observations_gprof).any("channel_gprof").data, colors="k", linestyles=["--"])

lon_ticks = np.arange(lons.min() // 5 * 5, lons.max() // 5 * 5 + 1, 5)
lat_ticks = np.arange(lats.min() // 5 * 5, lats.max() // 5 * 5 + 1, 5)

add_ticks(ax, lons=lon_ticks, lats=lat_ticks)
ax.coastlines(color="grey")

patch_legend = Patch(facecolor=swath, edgecolor='none', label='GPROF retrieval database')
line_legend = Line2D([0], [0], color='C3', linewidth=2, label='AMSR2 swath', linestyle="--")
# Add the legend to the plot
ax.legend(handles=[line_legend, patch_legend], loc='lower left')

ax.set_title("GMI and DPR swaths")
fig.savefig("gmi_sims.png", dpi=200, bbox_inches="tight")

In [ ]:
np.round(

In [ ]:
plt.pcolormesh(atms_observations.observations_gprof.data[..., 5])
plt.colorbar()

In [ ]:
plt.pcolormesh(atms_observations.observations_gprof.data[..., -3])
plt.colorbar()

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 6

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = (tbs_sim >= 0.0) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1], np.mean((tbs_sim[valid] - tbs_ref[valid]) ** 2))
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = (tbs_sim >= 0) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1], np.mean((tbs[valid] - tbs_ref[valid]) ** 2))

ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 4

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1], np.mean((tbs_sim[valid] - tbs_ref[valid]) ** 2))
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1], np.mean((tbs[valid] - tbs_ref[valid]) ** 2))

ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 0

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4)
crs = ccrs.PlateCarree()

chan = 4

ax = fig.add_subplot(gs[0, 0], projection=crs)
lons = atms_observations.longitude.data
lats = atms_observations.latitude.data
tbs_ref = atms_observations.observations_gprof[..., chan].data
ax.pcolormesh(lons, lats, atms_observations.observations_gprof[..., chan])

ax = fig.add_subplot(gs[0, 1], projection=crs)
tbs_sim = sim_data_r.simulated_tbs[..., chan].data
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs_sim[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs_sim - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

ax = fig.add_subplot(gs[0, 2], projection=crs)
bias = sim_data_r.brightness_temperature_biases[..., chan]
ax.pcolormesh(lons, lats, tbs_sim - bias)

ax = fig.add_subplot(gs[0, 3], projection=crs)
tbs = sim_data_r.satformer_tbs.data[..., chan]
valid = np.isfinite(tbs_sim) * np.isfinite(tbs_ref)
print(np.corrcoef(tbs[valid], tbs_ref[valid])[0, 1])
ax.pcolormesh(lons, lats, tbs - tbs_ref, vmin=-10, vmax=10, cmap="coolwarm")

## Run Satformer

Below we produce synthetic Tbs using the Satformer model trained on Tb collocations between GMI, ATMS and AMSR2

In [ ]:
from pansat import TimeRange
from pansat.catalog import Index
from pansat.products.satellite.gpm import l1c_r_gpm_gmi, l1c_noaa20_atms
from pansat.environment import get_index
from pansat.catalog.index import find_matches
from gprof_nn.data.pretraining import InputLoader
from pytorch_retrieve.architectures import load_and_compile_model, load_model
from pytorch_retrieve import InferenceConfig
from pytorch_retrieve.config import RetrievalOutputConfig
from pytorch_retrieve.inference import run_inference
from pytorch_retrieve.retrieval_output import ExpectedValue

output_config = RetrievalOutputConfig(model.output_config["output_observations"], "ExpectedValue", {})
retrieval_output = {"output_observations": {"output_observations": output_config}}
inference_config = InferenceConfig(tile_size=128, spatial_overlap=32, retrieval_output=retrieval_output)

def run_satformer(input_data, area) -> xr.Dataset:
    """
    Run satformer for given collocation scene and resample the results.
    """
    time_range = TimeRange(input_data.scan_time.mean().item())
    gmi_recs = l1c_r_gpm_gmi.get(time_range=time_range)
    atms_recs = l1c_noaa20_atms.get(time_range=time_range)
    gmi_index = Index.index(l1c_r_gpm_gmi, gmi_recs)
    atms_index = Index.index(l1c_noaa20_atms, atms_recs)
    matches = find_matches(gmi_index, atms_index)
    input_loader = InputLoader(matches)
    inpt, fname, aux = input_loader.load_data(0)
    
    results = run_inference(
        model,
        input_loader,
        inference_config,
        exclude_from_tiling=[
            "input_observation_mask",
            "two_meter_temperature_mask",
            "total_column_water_vapor_mask",
            "land_fraction_mask",
            "ice_fraction_mask",
            "leaf_area_index_mask",
            "elevation_mask",
            "ir_observations_mask",
        ],
    )
    
    results[0]["latitude"] = (("x", "y"), aux["latitude"].data)
    results[0]["longitude"] = (("x", "y"), aux["longitude"].data)
    invalid = np.isnan(inpt["output_observation_props"].numpy()).any(1)[0, 0]
    results[0]["output_observations"].data[..., invalid] = np.nan
    results_r = resample_data(results[0].transpose("x", "y", ...), area, radius_of_influence=15e3)

    return results_r

In [ ]:
sf_data_r = run_satformer(atms_observations, area)

In [ ]:
sf_data_r

In [ ]:
plt.pcolormesh(sf_data_r.output_observations[..., 7])
plt.colorbar()

## Collect data from collocations

Below, we iterate over all collocations and extract data from pixels with valid simulator data.

In [ ]:
from tqdm import tqdm
tbs_actual = []
tbs_sim = []
tbs_sim_bias = []
tbs_sf = []
eia = []
surface_type = []
surface_precip = []

for colloc in tqdm(np.random.permutation(collocations)[:100]):
    
    atms_observations = xr.load_dataset(colloc, group="input_data")
    lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
    area = SwathDefinition(lats=lats, lons=lons)

    reference_data = xr.load_dataset(colloc, group="reference_data")
    granule = get_granule(reference_data)
    
    sim_data_r = load_and_resample_sim_data(granule, atms_observations, area)
    valid = (
        (atms_observations.observations_gprof.data > -1000).all(axis=-1) *
        (sim_data_r.simulated_tbs.data > -1000).all(axis=-1) *
        (sim_data_r.brightness_temperature_biases.data > -1000).all(axis=-1)
    )
    
    tbs_actual.append(atms_observations.observations_gprof.data[valid])
    tbs_sim.append(sim_data_r.simulated_tbs.data[valid])
    tbs_sim_bias.append(sim_data_r.brightness_temperature_biases.data[valid])
    tbs_sf.append(sim_data_r.satformer_tbs.data[valid])
    eia.append(atms_observations.earth_incidence_angle.data[valid, 0])
    surface_type.append(sim_data_r.surface_type.data[valid])
    surface_precip.append(sim_data_r.surface_precip.data[valid])


In [ ]:
atms_observations.observations_gprof

In [ ]:
results = xr.Dataset({
    "tbs_actual":  (("samples", "channels"), np.concatenate(tbs_actual)),
    "tbs_sim": (("samples", "channels"), np.concatenate(tbs_sim)),
    "tbs_sim_bias": (("samples", "channels"), np.concatenate(tbs_sim_bias)),
    "tbs_sf": (("samples", "channels"), np.concatenate(tbs_sf)),
    "eia": (("samples"), np.concatenate(eia)),
    "surface_type":  (("samples",), np.concatenate(surface_type)),
})


In [ ]:
np.isfinite(results["tbs_sim_bias"].data[..., -4]).any()

In [ ]:
from matplotlib.gridspec import GridSpec

CHANNELS = [
    "10 GHz",
    "10 GHz",
    "18 GHz",
    "18 GHz",
    "23 GHz",
    "23 GHz",
    "37 GHz",
    "37 GHz",
    "89 GHz",
    "89 GHz",
]

def make_scater_plots(results):
    fig = plt.figure(figsize=(20, 50))
    gs = GridSpec(10, 5, width_ratios=[0.3, 1.0, 1.0, 1.0, 1.0])
    
    
    for chan in range(10):
        
        ax = fig.add_subplot(gs[chan, 0])
        ax.set_axis_off()
        ax.text(0, 0, CHANNELS[chan], rotation=90, ha="center", va="center")
        ax.set_ylim(-2, 2)
        
        tbs_act = results["tbs_actual"].data[..., chan]
        tbs_sim = results["tbs_sim"].data[..., chan]
        tbs_bias = results["tbs_sim_bias"].data[..., chan]
        tbs_sf = results["tbs_sf"].data[..., chan]
        eia = results["eia"].data
        
        #
        # Simulated TBS
        #
        
        ax = fig.add_subplot(gs[chan, 1])
        if chan == 0:
            ax.set_title("Simulated", loc="center")
        bins = np.linspace(tbs_act.min(), tbs_act.max())
        tbs = tbs_sim
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")

        ax.set_ylabel("Simulated $T_b$ [K]")
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Bias-corrected, simulated TBs
        #
        
        ax = fig.add_subplot(gs[chan, 2])
        if chan == 0:
            ax.set_title("Simulated - Bias", loc="center")
        tbs = tbs_sim - tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # EIA adapted bias correction
        #
        
        ax = fig.add_subplot(gs[chan, 3])
        if chan == 0:
            ax.set_title(r"Simulated - $\frac{\cos(\theta_{\text{GMI}})}{\cos(\theta)}$ Bias", loc="center")
        tbs = tbs_sim - np.cos(np.deg2rad(48.0)) / np.cos(np.deg2rad(eia)) * tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Satformer results
        #
        
        ax = fig.add_subplot(gs[chan, 4])
        if chan == 0:
            ax.set_title(r"Satformer", loc="center")
        
        tbs = tbs_sf
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
                                 
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        #ax.text(0.1 * x[0], 0.9 * x[-1], f"Bias:  {bias:.2f}\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}")
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)

    return fig, ax

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

## Test simulations

In [ ]:
time_range = TimeRange("2020-06-03", "2020-06-04")
gmi_recs = l1c_r_gpm_gmi.get(time_range=time_range)
atms_recs = l1c_noaa20_atms.get(time_range=time_range)
gmi_index = Index.index(l1c_r_gpm_gmi, gmi_recs)
atms_index = Index.index(l1c_noaa20_atms, atms_recs)
matches = find_matches(gmi_index, atms_index)
input_loader = InputLoader(matches)

In [ ]:
inpt, fname, aux = input_loader.load_data(1)

In [ ]:
inpt["observations"].shape

In [ ]:
plt.pcolormesh(inpt["observations"][0, 2, 6])

In [ ]:
inpt.keys()

In [ ]:
tile = {name: tensor[..., 350:478, 20:148] for name, tensor in inpt.items()}
for name, tensor in inpt.items():
    if name.endswith("_mask"):
        tile[name] = inpt[name]
tbs_targ = aux["target_observations"].data[..., 64:128, 64:128]

In [ ]:
tile["observations"].shape

In [ ]:
plt.imshow(tile["observations"][0, 2, 0])

In [ ]:
import torch

with torch.no_grad():
    y_pred = model(tile)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

In [ ]:
from copy import deepcopy

props = tile["output_observation_props"].clone()
props[:, :, 1:] = props[:, :, :1]
props[0, 3, :] = torch.tensor(np.linspace(1.75, 5.2, 9))[..., None, None]
props[0, 4, :] = torch.tensor(np.linspace(416e3, 800e3, 9))[..., None, None]

tile_bw = deepcopy(tile)
tile_bw["output_observation_props"] = props

In [ ]:

with torch.no_grad():
    y_pred = model(tile_bw)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

## Beam width

In [ ]:
def simulate_beam_width(beam_width):
    
    props = tile["output_observation_props"].clone()[:, :, :1]
    props[0, 3, :] = beam_width
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"89 GHz, beam width = {beam_width:.2f} deg.")
    plt.imshow(y_pred[0], vmin=160, vmax=240)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)


In [ ]:
model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

interact(simulate_beam_width, beam_width=(1.5, 5.2))


## WV-channel offset

In [ ]:
def simulate_offset(offset):
    
    props = tile["output_observation_props"].clone()[:, :, -1:]
    props[0, 1, :] = offset
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"183 +/- {offset:.2f} GHz")
    plt.imshow(y_pred[0], vmin=240, vmax=270)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_offset, offset=(1.0, 7.0))

## EIA

In [ ]:
def simulate_eia(eia):
    
    props = tile["output_observation_props"].clone()[:, :, [-5]]
    props[0, -2, :] += eia
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(rf"183 +/- 7 GHz, $\Delta\theta = ${eia:.2f}")
    plt.imshow(y_pred[0], vmin=220, vmax=280)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_eia, eia=(-20.0, 20.0))

In [ ]:
plt.imshow(props[0, -2, -1])
plt.colorbar()

## Run simul